In [6]:
import os 
from dotenv import load_dotenv

load_dotenv()

#os.environ["FIRECRAWL_API_KEY"] = os.getenv("FIRECRAWL_API")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_TOKEN")

#FIRECRAWL_API = os.environ["FIRECRAWL_API_KEY"]
GROQ_API = os.environ["GROQ_API_KEY"]
HF_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]



In [ ]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(
    query="Batman",
    load_max_docs=10,
    doc_content_chars_max=10000
)

docs = loader.load()

In [ ]:
for doc in docs:
        print(doc.page_content)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100,)
chunks=text_splitter.split_documents(docs)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
hg_embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_chroma import Chroma
Vector_DB = Chroma.from_documents(
    documents=chunks,
    embedding=hg_embedding,
    collection_name="Batman_DB",
    persist_directory="Chroma_DB"
)

In [10]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

hg_embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Vector_DB = Chroma(
    collection_name="Batman_DB",
    embedding_function=hg_embedding,
    persist_directory="Chroma_DB"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
results= Vector_DB.similarity_search(query="gotham city")
for result in results:
    print(result.page_content)

In [8]:
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage


In [9]:
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024,api_key=GROQ_API)
messages=[
    HumanMessage(content="Who is batman")
]
parser=StrOutputParser()

chain =  llm | parser
response= chain.invoke(messages)

print(response)


Batman is a fictional superhero appearing in American comic books published by DC Comics. He is one of the most iconic and recognizable characters in the world of comics and popular culture.

**Origin Story:**
Batman's real name is Bruce Wayne, a billionaire philanthropist and owner of Wayne Enterprises. As a child, Bruce witnessed his parents, Thomas and Martha Wayne, being murdered in front of him in a dark alley in Gotham City. This traumatic event drove Bruce to dedicate his life to fighting crime and protecting the innocent.

**Costume and Abilities:**
Batman's alter ego is a crime-fighter who wears a distinctive costume, including a black cape, cowl, and Batsuit. He is a skilled martial artist, detective, and strategist, using his intellect and physical abilities to outsmart and defeat his enemies. He also uses various gadgets and tools, such as his trusty Batmobile, grappling hook, and batarangs.

**Personality:**
Batman is a complex character with a dark and brooding personalit

In [ ]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage

message=[
    SystemMessage(content="You are an expert psychologist specializing in fictional characters.Your role is to explain characters using evidence, clear reasoning, and psychological analysis.Be objective, concise, and avoid making up facts. If information is uncertain, say so instead of guessing."),
    HumanMessage(content="Who is batman")
]
response=chain.invoke(message)
print(response)

In [ ]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage

message=[
    SystemMessage(content="""
                   You are a joker  superhater.
                   Answer with enthusiasm and use a casual tone.
                   in one line as response in simple english
                  """),
    HumanMessage(content="Who is joker")
]
response=chain.invoke(message)
print(response)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}."),
    ("human", "{text}")

])

chain=prompt_template | llm | parser

response = chain.invoke({
    "input_language":"english",
    "output_language":"tamil",
    "text":"batman"
})
print(response)


In [13]:
from langchain_core.messages import HumanMessage,AIMessage

history=[
    HumanMessage("Hi my name is loki"),
    AIMessage("Greetings, Loki, I'm here to assist you with any questions or tasks you may have."),
    HumanMessage("From now your name is jarvis"),
    AIMessage("I'm at your service, master.")

]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_groq import ChatGroq


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that return repsonse to user in one line!"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{text}")
])

llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024,api_key=GROQ_API)

chain=prompt_template | llm | parser

response = chain.invoke({
     "history":history,
    "text":"what is my name and your name"

})
print(response)


KeyError: "Input to ChatPromptTemplate is missing variables {'history'}.  Expected: ['history', 'text'] Received: ['text']\nNote: if you intended {history} to be part of the string and not a variable, please escape it with double curly braces like: '{{history}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "